In [ ]:
#!pip install numpy matplotlib scikit-learn torch
#!pip install phe tenseal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim

import tenseal as ts

import time

In [ ]:

# δημιουργία synthetic dataset
X, y = make_moons(n_samples=1500, noise=0.20, random_state=42)

#  split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# standardization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# scatter plot
plt.figure(figsize=(6, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, alpha=0.7)
plt.title("Synthetic make_moons data")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
y_test_t = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
def compute_accuracy(model, X_t, y_np):
    model.eval()
    with torch.no_grad():
        logits = model(X_t)
        probs = torch.sigmoid(logits).cpu().numpy().reshape(-1)
        preds = (probs >= 0.5).astype(int)
    return accuracy_score(y_np, preds)


def train_model(model, X_train_t, y_train_t, X_test_t, y_test_np,
                epochs=100, lr=1e-3, batch_size=64):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_acc_hist = []
    test_acc_hist = []
    loss_hist = []

    n = X_train_t.shape[0]

    for epoch in range(epochs):
        model.train()

        perm = torch.randperm(n)
        X_shuf = X_train_t[perm]
        y_shuf = y_train_t[perm]

        epoch_losses = []

        for start in range(0, n, batch_size):
            end = start + batch_size

            xb = X_shuf[start:end]
            yb = y_shuf[start:end]

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        train_acc = compute_accuracy(model, X_train_t, y_train_t.numpy().reshape(-1))
        test_acc = compute_accuracy(model, X_test_t, y_test_np)

        train_acc_hist.append(train_acc)
        test_acc_hist.append(test_acc)
        loss_hist.append(np.mean(epoch_losses))

    return {
        "model": model,
        "train_acc": train_acc_hist,
        "test_acc": test_acc_hist,
        "loss": loss_hist
    }

In [ ]:
original_model = BaselineMLP()

original_result = train_model(
    original_model,
    X_train_t,
    y_train_t,
    X_test_t,
    y_test,
    epochs=120,
    lr=1e-3,
    batch_size=64
)

print("Final train accuracy:", original_result["train_acc"][-1])
print("Final test accuracy:", original_result["test_acc"][-1])

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(original_result["train_acc"], label="Train accuracy")
plt.plot(original_result["test_acc"], label="Test accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("model accuracy")
plt.legend()
plt.show()

In [ ]:
class SquareActivation(nn.Module):
    def forward(self, x):
        return x * x


class CryptoNetMini(nn.Module):
    def __init__(self, hidden_dim=16):
        super().__init__()
        self.fc1 = nn.Linear(2, hidden_dim)
        self.act = SquareActivation()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.fc2(x)
        return x

In [ ]:
central_crypto_model = CryptoNetMini(hidden_dim=16)

crypto_result = train_model(
    original_model,
    X_train_t,
    y_train_t,
    X_test_t,
    y_test,
    epochs=120,
    lr=1e-3,
    batch_size=64
)

print("Final train accuracy:", crypto_result["train_acc"][-1])
print("Final test accuracy:", crypto_result["test_acc"][-1])

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(crypto_result["train_acc"], label="Train accuracy")
plt.plot(crypto_result["test_acc"], label="Test accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CryptoNet-style model accuracy")
plt.legend()
plt.show()

In [ ]:
def plot_decision_boundary(model, X, y, title):
    model.eval()

    x_min, x_max = X[:, 0].min() - 0.6, X[:, 0].max() + 0.6
    y_min, y_max = X[:, 1].min() - 0.6, X[:, 1].max() + 0.6

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_t = torch.tensor(grid, dtype=torch.float32)

    with torch.no_grad():
        logits = model(grid_t).cpu().numpy().reshape(xx.shape)
        probs = 1 / (1 + np.exp(-logits))

    plt.figure(figsize=(6, 5))
    plt.contourf(xx, yy, probs, levels=20, alpha=0.65)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", alpha=0.75)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.show()

In [ ]:
plot_decision_boundary(
    original_model,
    X_test,
    y_test,
    "Original model decision boundary"
)

plot_decision_boundary(
    central_crypto_model,
    X_test,
    y_test,
    "CryptoNet-style decision boundary"
)

In [ ]:
def create_ckks_context():
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=16384,
        coeff_mod_bit_sizes=[60, 40, 40, 40, 40, 60]
    )
    context.global_scale = 2 ** 40
    context.generate_galois_keys()
    context.generate_relin_keys()
    return context


secret_context = create_ckks_context()

public_context = ts.context_from(secret_context.serialize())
public_context.make_context_public()

In [ ]:
def extract_model_weights(model):
    model.eval()

    W1 = model.fc1.weight.detach().cpu().numpy()
    b1 = model.fc1.bias.detach().cpu().numpy()

    W2 = model.fc2.weight.detach().cpu().numpy()
    b2 = model.fc2.bias.detach().cpu().numpy()

    return W1, b1, W2, b2


W1, b1, W2, b2 = extract_model_weights(central_crypto_model)

print("W1 shape:", W1.shape)
print("b1 shape:", b1.shape)
print("W2 shape:", W2.shape)
print("b2 shape:", b2.shape)

Plaintext forward pass για ένα δείγμα

In [ ]:
sample_id = 0

x_sample = X_test[sample_id]
y_true = y_test[sample_id]

with torch.no_grad():
    plain_logit = central_crypto_model(
        torch.tensor(x_sample.reshape(1, -1), dtype=torch.float32)
    ).item()

plain_prob = 1 / (1 + np.exp(-plain_logit))
plain_pred = int(plain_prob >= 0.5)

print("True label:", y_true)
print("Plain logit:", plain_logit)
print("Plain probability:", plain_prob)
print("Plain prediction:", plain_pred)

Encrypted inference με TenSEAL

Enc(x)→Enc(W1 * ​x + b1​)→Enc(( W1 * ​x + b1​)^2)→Enc(W2​ * a1​ + b2​)


In [ ]:
def encrypted_cryptonet_inference(x, W1, b1, W2, b2, context):
    enc_x = ts.ckks_vector(context, x.tolist())

    hidden_encrypted = []

    for neuron_idx in range(W1.shape[0]):
        z = enc_x.dot(W1[neuron_idx].tolist())
        z = z + [float(b1[neuron_idx])]
        a = z * z          # square activation
        hidden_encrypted.append(a)

    enc_out = hidden_encrypted[0] * float(W2[0, 0])

    for j in range(1, W2.shape[1]):
        enc_out = enc_out + hidden_encrypted[j] * float(W2[0, j])

    enc_out = enc_out + [float(b2[0])]
    return enc_out

In [ ]:
enc_logit = encrypted_cryptonet_inference(
    x_sample,
    W1,
    b1,
    W2,
    b2,
    secret_context
)

decrypted_logit = enc_logit.decrypt()[0]
decrypted_prob = 1 / (1 + np.exp(-decrypted_logit))
decrypted_pred = int(decrypted_prob >= 0.5)

print("True label:", y_true)
print("Plain logit:", plain_logit)
print("Encrypted/decrypted logit:", decrypted_logit)
print("Absolute error:", abs(plain_logit - decrypted_logit))
print("Encrypted/decrypted probability:", decrypted_prob)
print("Encrypted/decrypted prediction:", decrypted_pred)

Encrypted inference σε πολλά test samples

In [ ]:
def evaluate_encrypted_inference(model, X_eval, y_eval, context, max_samples=100):
    W1, b1, W2, b2 = extract_model_weights(model)

    n = min(max_samples, len(X_eval))

    plain_logits = []
    encrypted_logits = []
    encrypted_preds = []

    start = time.perf_counter()

    for i in range(n):
        x = X_eval[i]

        with torch.no_grad():
            plain_logit_i = model(
                torch.tensor(x.reshape(1, -1), dtype=torch.float32)
            ).item()

        enc_logit_i = encrypted_cryptonet_inference(
            x, W1, b1, W2, b2, context
        )

        dec_logit_i = enc_logit_i.decrypt()[0]
        dec_prob_i = 1 / (1 + np.exp(-dec_logit_i))
        dec_pred_i = int(dec_prob_i >= 0.5)

        plain_logits.append(plain_logit_i)
        encrypted_logits.append(dec_logit_i)
        encrypted_preds.append(dec_pred_i)

    end = time.perf_counter()

    acc = accuracy_score(y_eval[:n], encrypted_preds)
    avg_abs_error = np.mean(np.abs(np.array(plain_logits) - np.array(encrypted_logits)))

    return {
        "accuracy": acc,
        "avg_abs_error": avg_abs_error,
        "runtime": end - start,
        "plain_logits": np.array(plain_logits),
        "encrypted_logits": np.array(encrypted_logits),
        "encrypted_preds": np.array(encrypted_preds)
    }

In [ ]:
encrypted_eval = evaluate_encrypted_inference(
    central_crypto_model,
    X_test,
    y_test,
    secret_context,
    max_samples=100
)

print("Encrypted inference accuracy on subset:", encrypted_eval["accuracy"])
print("Average absolute logit error:", encrypted_eval["avg_abs_error"])
print("Runtime seconds:", encrypted_eval["runtime"])

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(
    encrypted_eval["plain_logits"],
    encrypted_eval["encrypted_logits"],
    alpha=0.75
)

min_v = min(encrypted_eval["plain_logits"].min(), encrypted_eval["encrypted_logits"].min())
max_v = max(encrypted_eval["plain_logits"].max(), encrypted_eval["encrypted_logits"].max())

plt.plot([min_v, max_v], [min_v, max_v], linestyle="--")
plt.xlabel("Plain logits")
plt.ylabel("Encrypted/decrypted logits")
plt.title("Plain vs encrypted inference logits")
plt.show()